In [1]:
!pip install duckdb

   ---------------------------------------- 0.0/13.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/13.2 MB ? eta -:--:--
   ---------------------------------------- 0.1/13.2 MB 812.7 kB/s eta 0:00:17
    --------------------------------------- 0.2/13.2 MB 2.0 MB/s eta 0:00:07
   - -------------------------------------- 0.4/13.2 MB 2.3 MB/s eta 0:00:06
   - -------------------------------------- 0.5/13.2 MB 2.3 MB/s eta 0:00:06
   - -------------------------------------- 0.6/13.2 MB 2.5 MB/s eta 0:00:06
   -- ------------------------------------- 0.7/13.2 MB 2.5 MB/s eta 0:00:05
   -- ------------------------------------- 0.8/13.2 MB 2.3 MB/s eta 0:00:06
   -- ------------------------------------- 0.9/13.2 MB 2.3 MB/s eta 0:00:06
   -- ------------------------------------- 0.9/13.2 MB 2.3 MB/s eta 0:00:06
   --- ------------------------------------ 1.0/13.2 MB 2.1 MB/s eta 0:00:06
   --- ------------------------------------ 1.1/13.2 MB 2.1 MB/s eta 0:00:06
   --- ----


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
!pip install numpy

   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.6 MB 653.6 kB/s eta 0:00:20
   ---------------------------------------- 0.1/12.6 MB 1.4 MB/s eta 0:00:09
    --------------------------------------- 0.2/12.6 MB 1.5 MB/s eta 0:00:09
    --------------------------------------- 0.2/12.6 MB 1.1 MB/s eta 0:00:11
    --------------------------------------- 0.3/12.6 MB 1.3 MB/s eta 0:00:10
   - -------------------------------------- 0.3/12.6 MB 1.2 MB/s eta 0:00:11
   - -------------------------------------- 0.4/12.6 MB 1.2 MB/s eta 0:00:11
   - -------------------------------------- 0.4/12.6 MB 1.1 MB/s eta 0:00:12
   - -------------------------------------- 0.5/12.6 MB 1.0 MB/s eta 0:00:12
   - -------------------------------------- 0.5/12.6 MB 1.1 MB/s eta 0:00:11
   -- ------------------------------------- 0.6/12.6 MB 1.2 MB/s eta 0:00:10
   -- -----


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
!pip install pandas

   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.1/9.7 MB 1.3 MB/s eta 0:00:08
    --------------------------------------- 0.2/9.7 MB 1.5 MB/s eta 0:00:07
   - -------------------------------------- 0.3/9.7 MB 1.8 MB/s eta 0:00:06
   - -------------------------------------- 0.3/9.7 MB 1.6 MB/s eta 0:00:06
   - -------------------------------------- 0.4/9.7 MB 1.6 MB/s eta 0:00:06
   - -------------------------------------- 0.4/9.7 MB 1.4 MB/s eta 0:00:07
   - -------------------------------------- 0.4/9.7 MB 1.3 MB/s eta 0:00:08
   - -------------------------------------- 0.4/9.7 MB 1.3 MB/s eta 0:00:08
   - -------------------------------------- 0.4/9.7 MB 1.3 MB/s eta 0:00:08
   - -------------------------------------- 0.5/9.7 MB 972.0 kB/s eta 0:00:10
   - -------------------------------------- 0.5/9.7 MB 972.0 kB/s eta 0:00:10
   - -----------------


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import duckdb
con = duckdb.connect()
p = "c2b_step4_kept_prices.parquet"

print(con.sql(f"DESCRIBE SELECT * FROM '{p}'").df())
print(con.sql(f"SELECT COUNT(*) AS n FROM '{p}'").df())
print(con.sql(f"SELECT * FROM '{p}' LIMIT 5").df().T)

                column_name column_type null   key default extra
0                 rate_file     VARCHAR  YES  None    None  None
1                       tin     VARCHAR  YES  None    None  None
2                       npi     VARCHAR  YES  None    None  None
3                  tin_type     VARCHAR  YES  None    None  None
4             business_name     VARCHAR  YES  None    None  None
5               entry_label     VARCHAR  YES  None    None  None
6                    county     VARCHAR  YES  None    None  None
7         provider_group_id     INTEGER  YES  None    None  None
8         billing_code_type     VARCHAR  YES  None    None  None
9              billing_code     VARCHAR  YES  None    None  None
10   additional_information     VARCHAR  YES  None    None  None
11  negotiation_arrangement     VARCHAR  YES  None    None  None
12          negotiated_type     VARCHAR  YES  None    None  None
13            billing_class     VARCHAR  YES  None    None  None
14                  setti

In [7]:
import duckdb
con = duckdb.connect()
p = "c2b_step4_kept_prices.parquet"
for col in ["entry_label", "copy_status", "negotiated_type", "billing_class", "billing_code_type"]:
    print(con.sql(f"SELECT {col}, COUNT(*) n FROM '{p}' GROUP BY ALL ORDER BY n DESC").df(), "\n")
print(con.sql(f"""
  SELECT COUNT(DISTINCT tin) tins, COUNT(DISTINCT npi) npis,
         COUNT(DISTINCT billing_code) codes,
         SUM(billing_code_modifier IS NOT NULL) with_modifier
  FROM '{p}'
""").df())

             entry_label      n
0                   none  50369
1  direct_ffs_commercial    343
2                    ppo    182
3                    ipa     29
4         direct_network     10 

             copy_status      n
0                 single  26042
1       identical_copies  22059
2       differing_copies   2706
3  within_group_variants    126 

  negotiated_type      n
0      negotiated  50933 

   billing_class      n
0  institutional  35978
1   professional  14955 

  billing_code_type      n
0                RC  35978
1               CPT  12685
2             HCPCS   2270 

   tins  npis  codes  with_modifier
0  2856  3694    768         5998.0


In [11]:
import duckdb

KEPT = "c2b_step4_kept_prices.parquet"   # adjust if the file is in another folder
con = duckdb.connect()

import os
print(os.getcwd())
print([f for f in os.listdir() if f.endswith(".parquet")])

d:\Hilabs
['c2b_step4_kept_prices.parquet']


In [15]:
import os
KEPT = r"d:\\Hilabs\\c2b_step4_kept_prices.parquet"   # change to your real path
print(os.path.exists(KEPT))

True


In [14]:
import glob
print(os.getcwd())
print(glob.glob(r"d:\Hilabs\**\c2b_step4_kept_prices.parquet", recursive=True))

d:\Hilabs
['d:\\Hilabs\\c2b_step4_kept_prices.parquet']


In [18]:
import os, duckdb
KEPT = r"d:\\Hilabs\\c2b_step4_kept_prices.parquet"   # change to your real path
print(os.path.exists(KEPT))
con = duckdb.connect()

True


In [19]:
for col in ["entry_label", "copy_status", "negotiated_type", "billing_class", "billing_code_type"]:
    print(con.sql(f"SELECT {col}, COUNT(*) n FROM '{KEPT}' GROUP BY ALL ORDER BY n DESC").df(), "\n")
print(con.sql(f"""
  SELECT COUNT(DISTINCT tin) tins, COUNT(DISTINCT npi) npis,
         COUNT(DISTINCT billing_code) codes,
         SUM(billing_code_modifier IS NOT NULL) with_modifier
  FROM '{KEPT}'
""").df())

             entry_label      n
0                   none  50369
1  direct_ffs_commercial    343
2                    ppo    182
3                    ipa     29
4         direct_network     10 

             copy_status      n
0                 single  26042
1       identical_copies  22059
2       differing_copies   2706
3  within_group_variants    126 

  negotiated_type      n
0      negotiated  50933 

   billing_class      n
0  institutional  35978
1   professional  14955 

  billing_code_type      n
0                RC  35978
1               CPT  12685
2             HCPCS   2270 

   tins  npis  codes  with_modifier
0  2856  3694    768         5998.0


In [20]:
con.sql(f"""
CREATE OR REPLACE TABLE cand AS
SELECT *, row_number() OVER () AS price_id,
  CASE entry_label
    WHEN 'ppo' THEN 0
    WHEN 'direct_ffs_commercial' THEN 1
    WHEN 'direct_network' THEN 1
    WHEN 'none' THEN 2
    WHEN 'ipa' THEN 3
    ELSE 4 END AS network_rank,
  CASE WHEN billing_code_modifier IS NULL OR billing_code_modifier = '' THEN 0 ELSE 1 END AS modifier_rank,
  CASE WHEN service_code = '11' THEN 0 WHEN service_code = '22' THEN 1 ELSE 2 END AS pos_rank,
  abs(ln(greatest(negotiated_rate, 0.01))
    - ln(greatest(median(negotiated_rate) OVER (PARTITION BY billing_code_type, billing_code), 0.01))) AS median_dist
FROM '{KEPT}'
WHERE negotiated_rate > 0
""")

con.sql("""
CREATE OR REPLACE TABLE ranked AS
SELECT *,
  row_number() OVER (PARTITION BY tin, npi, billing_code_type, billing_code
    ORDER BY network_rank, modifier_rank, pos_rank, median_dist, price_id) AS rn,
  count(DISTINCT negotiated_rate) OVER (PARTITION BY tin, npi, billing_code_type, billing_code) AS n_distinct_rates,
  min(negotiated_rate) OVER (PARTITION BY tin, npi, billing_code_type, billing_code) AS min_rate,
  max(negotiated_rate) OVER (PARTITION BY tin, npi, billing_code_type, billing_code) AS max_rate
FROM cand
""")

con.sql("COPY (SELECT * FROM ranked WHERE rn=1) TO 'c4_chosen.parquet' (FORMAT parquet)")
con.sql("COPY (SELECT * FROM ranked WHERE rn>1) TO 'c4_alternatives.parquet' (FORMAT parquet)")

print(con.sql("SELECT count(*) AS candidates FROM cand").df())
print(con.sql("""
  SELECT count(*) AS chosen,
         sum(n_distinct_rates > 1) AS keys_with_conflicting_rates
  FROM ranked WHERE rn = 1""").df())
print(con.sql("SELECT count(*) AS zero_or_neg_dropped FROM read_parquet('%s') WHERE negotiated_rate <= 0" % KEPT).df())

   candidates
0       50933
   chosen  keys_with_conflicting_rates
0   21614                       3009.0
   zero_or_neg_dropped
0                    0


In [21]:
# 1. Which conflicts remain, and how large are they?
print(con.sql("""
  SELECT billing_code_type, copy_status,
         count(*) AS keys, round(median(max_rate/min_rate),2) AS median_spread,
         round(max(max_rate/min_rate),2) AS max_spread
  FROM ranked WHERE rn=1 AND n_distinct_rates > 1
  GROUP BY ALL ORDER BY keys DESC""").df())

# 2. Which rule actually decided each conflict?
print(con.sql("""
  WITH t AS (
    SELECT *, 
      count(DISTINCT network_rank)  OVER w AS nr,
      count(DISTINCT modifier_rank) OVER w AS mr,
      count(DISTINCT pos_rank)      OVER w AS pr
    FROM cand
    WINDOW w AS (PARTITION BY tin, npi, billing_code_type, billing_code))
  SELECT CASE WHEN nr>1 THEN '1 network' WHEN mr>1 THEN '2 modifier'
              WHEN pr>1 THEN '3 place of service' ELSE '4 median tie-break' END AS decided_by,
         count(*) AS keys
  FROM (SELECT DISTINCT tin, npi, billing_code_type, billing_code, nr, mr, pr FROM t
        WHERE (SELECT count(DISTINCT negotiated_rate) FROM cand c
               WHERE c.tin=t.tin AND c.npi=t.npi AND c.billing_code=t.billing_code
                 AND c.billing_code_type=t.billing_code_type) > 1)
  GROUP BY ALL ORDER BY keys DESC""").df())

  billing_code_type            copy_status  keys  median_spread  max_spread
0                RC       identical_copies   960           1.64     8700.99
1                RC                 single   707           1.35      518.50
2               CPT       differing_copies   580           1.36       14.04
3               CPT                 single   480           1.46        6.35
4                RC       differing_copies   142           1.16      306.14
5             HCPCS       differing_copies    43           1.68        5.51
6               CPT       identical_copies    36           3.56        5.00
7             HCPCS                 single    34           3.35       13.08
8                RC  within_group_variants    27           1.08        2.88
           decided_by  keys
0  3 place of service  1387
1  4 median tie-break  1195
2          2 modifier   427


In [22]:
print(con.sql("""
WITH k AS (
  SELECT tin, npi, billing_code_type, billing_code,
    count(DISTINCT negotiated_rate)                           AS n_rates,
    count(DISTINCT provider_group_id)                         AS n_groups,
    count(DISTINCT service_code)                              AS n_pos,
    count(DISTINCT coalesce(additional_information,''))       AS n_addl,
    count(DISTINCT coalesce(billing_code_modifier,''))        AS n_mod
  FROM cand GROUP BY ALL HAVING count(DISTINCT negotiated_rate) > 1)
SELECT billing_code_type,
       n_groups > 1 AS differs_by_group,
       n_pos    > 1 AS differs_by_pos,
       n_addl   > 1 AS differs_by_addl_info,
       n_mod    > 1 AS differs_by_modifier,
       count(*) AS keys
FROM k GROUP BY ALL ORDER BY keys DESC
""").df())

print(con.sql("""
SELECT additional_information, count(*) n FROM cand
WHERE billing_code_type='RC' GROUP BY ALL ORDER BY n DESC LIMIT 15
""").df())

   billing_code_type  differs_by_group  differs_by_pos  differs_by_addl_info  \
0                 RC              True            True                  True   
1                CPT              True           False                 False   
2                 RC              True           False                  True   
3                CPT              True            True                 False   
4                CPT              True            True                 False   
5                 RC             False           False                  True   
6                CPT             False            True                 False   
7                CPT              True           False                 False   
8                 RC              True           False                 False   
9                 RC              True            True                 False   
10               CPT             False           False                 False   
11             HCPCS              True  

In [23]:
KEY = "tin, npi, billing_code_type, billing_code, addl_key"

con.sql(f"""
CREATE OR REPLACE TABLE s0 AS
SELECT *, row_number() OVER () AS price_id,
  coalesce(additional_information,'') AS addl_key,
  CASE entry_label WHEN 'ppo' THEN 0
       WHEN 'direct_ffs_commercial' THEN 1 WHEN 'direct_network' THEN 1
       WHEN 'none' THEN 2 WHEN 'ipa' THEN 3 ELSE 4 END AS network_rank,
  CASE WHEN coalesce(billing_code_modifier,'') = '' THEN 0 ELSE 1 END AS modifier_rank,
  CASE WHEN service_code='11' THEN 0 WHEN service_code='22' THEN 1 ELSE 2 END AS pos_rank
FROM '{KEPT}' WHERE negotiated_rate > 0
""")

# 1-3: keep only the best-ranked rows per key at each stage
for i, col in enumerate(["network_rank", "modifier_rank", "pos_rank"], 1):
    con.sql(f"""CREATE OR REPLACE TABLE s{i} AS
        SELECT * FROM s{i-1}
        QUALIFY {col} = min({col}) OVER (PARTITION BY {KEY})""")

# 4: majority contract = the rate that appears in the most provider groups
con.sql(f"""
CREATE OR REPLACE TABLE s4 AS
WITH sup AS (
  SELECT {KEY}, negotiated_rate, count(DISTINCT provider_group_id) AS n_groups_with_rate
  FROM s3 GROUP BY ALL)
SELECT s3.*, sup.n_groups_with_rate
FROM s3 JOIN sup USING (tin, npi, billing_code_type, billing_code, addl_key, negotiated_rate)
QUALIFY n_groups_with_rate = max(n_groups_with_rate) OVER (PARTITION BY {KEY})
""")

# 5: still tied -> lowest rate (conservative, deterministic)
con.sql(f"""CREATE OR REPLACE TABLE s5 AS
    SELECT * FROM s4
    QUALIFY negotiated_rate = min(negotiated_rate) OVER (PARTITION BY {KEY})""")

# Final: one row per key
con.sql(f"""CREATE OR REPLACE TABLE chosen AS
    SELECT * FROM s5
    QUALIFY row_number() OVER (PARTITION BY {KEY} ORDER BY price_id) = 1""")
con.sql("CREATE OR REPLACE TABLE alternatives AS SELECT * FROM s0 WHERE price_id NOT IN (SELECT price_id FROM chosen)")

# Funnel: keys that still have more than one distinct rate after each stage
for name, label in [("s0","start"),("s1","after network"),("s2","after modifier"),
                    ("s3","after place of service"),("s4","after majority contract"),("s5","after lowest rate")]:
    n = con.sql(f"SELECT count(*) FROM (SELECT 1 FROM {name} GROUP BY {KEY} HAVING count(DISTINCT negotiated_rate) > 1)").fetchone()[0]
    print(f"{label:28s} {n}")

print(con.sql("SELECT count(*) AS chosen_rows FROM chosen").df())
con.sql("COPY chosen TO 'c4_chosen.parquet' (FORMAT parquet)")
con.sql("COPY alternatives TO 'c4_alternatives.parquet' (FORMAT parquet)")

start                        1382
after network                1382
after modifier               1092
after place of service       966
after majority contract      690
after lowest rate            0
   chosen_rows
0        27565


In [24]:
for i in range(6):
    con.sql(f"""CREATE OR REPLACE TABLE t{i} AS
        SELECT {KEY}, count(DISTINCT negotiated_rate) AS n{i},
               min(negotiated_rate) AS min{i}, max(negotiated_rate) AS max{i}
        FROM s{i} GROUP BY {KEY}""")

using = "tin, npi, billing_code_type, billing_code, addl_key"
con.sql(f"""
CREATE OR REPLACE TABLE key_audit AS
SELECT t0.*, t1.n1, t2.n2, t3.n3, t4.n4, t5.n5,
  CASE WHEN t0.n0 = 1 THEN 'no_conflict'
       WHEN t1.n1 = 1 THEN 'network'
       WHEN t2.n2 = 1 THEN 'modifier'
       WHEN t3.n3 = 1 THEN 'place_of_service'
       WHEN t4.n4 = 1 THEN 'majority_contract'
       ELSE 'lowest_rate_unresolved' END AS resolved_by,
  max0 / min0 AS spread
FROM t0 JOIN t1 USING ({using}) JOIN t2 USING ({using})
        JOIN t3 USING ({using}) JOIN t4 USING ({using}) JOIN t5 USING ({using})
""")

print(con.sql("""
  SELECT resolved_by, count(*) AS keys,
         round(median(spread),2) AS median_spread, round(max(spread),2) AS max_spread
  FROM key_audit GROUP BY ALL ORDER BY keys DESC""").df())

con.sql(f"""CREATE OR REPLACE TABLE chosen_flagged AS
    SELECT c.*, a.resolved_by, a.spread
    FROM chosen c JOIN key_audit a USING ({using})""")
con.sql("COPY chosen_flagged TO 'c4_chosen.parquet' (FORMAT parquet)")

              resolved_by   keys  median_spread  max_spread
0             no_conflict  26183           1.00        1.00
1  lowest_rate_unresolved    690           1.36       14.04
2                modifier    290           1.38       13.08
3       majority_contract    276           1.36       14.04
4        place_of_service    126           1.68        4.64


In [25]:
print(con.sql(f"""
  SELECT billing_code_type, 
         CASE WHEN spread < 1.1 THEN '<10%' WHEN spread < 1.5 THEN '10-50%'
              WHEN spread < 2 THEN '50-100%' ELSE '2x or more' END AS spread_band,
         count(*) AS keys
  FROM key_audit WHERE resolved_by = 'lowest_rate_unresolved'
  GROUP BY ALL ORDER BY keys DESC""").df())

print(con.sql("""
  SELECT s0.entry_label, count(DISTINCT (k.tin||k.npi||k.billing_code||k.addl_key)) AS keys
  FROM key_audit k JOIN s0 USING (tin, npi, billing_code_type, billing_code, addl_key)
  WHERE k.resolved_by = 'lowest_rate_unresolved'
  GROUP BY ALL ORDER BY keys DESC""").df())

   billing_code_type spread_band  keys
0                CPT      10-50%   221
1                CPT  2x or more   184
2                CPT        <10%   101
3                CPT     50-100%    73
4                 RC        <10%    32
5                 RC      10-50%    31
6              HCPCS  2x or more    17
7                 RC     50-100%    11
8                 RC  2x or more    10
9              HCPCS      10-50%     4
10             HCPCS     50-100%     4
11             HCPCS        <10%     2
             entry_label  keys
0                   none   689
1  direct_ffs_commercial     1


In [29]:
!python c4_select.py --kept "d:\Hilabs\c2b_step4_kept_prices.parquet" --out-dir outputs\c4

{
  "input_rows": 50933,
  "rows_dropped_rate_not_positive": 0,
  "candidate_rows": 50933,
  "chosen_keys": 27565,
  "alternative_rows": 23368,
  "unique_tins": 2856,
  "unique_npis": 3694,
  "keys_by_code_type": {
    "CPT": 10117,
    "HCPCS": 2023,
    "RC": 15425
  },
  "keys_resolved_by": {
    "modifier": 290,
    "majority_contract": 276,
    "no_conflict": 26183,
    "lowest_rate_unresolved": 690,
    "place_of_service": 126
  },
  "share_no_conflict": 0.9499,
  "share_unresolved_lowest_rate": 0.025,
  "unresolved_spread_median": 1.3601496725912068,
  "unresolved_spread_max": 14.036846722646947,
  "alternatives_by_reason": {
    "different_setting": 282,
    "modifier_variant": 613,
    "parallel_contract_alternative": 1352,
    "identical_rate": 21121
  },
  "funnel": [
    {
      "stage": "start (keys with more than one distinct rate)",
      "conflicting_keys": 1382
    },
    {
      "stage": "after network label",
      "conflicting_keys": 1382,
      "resolved_by_this_ru

In [27]:
!Get-ChildItem -Path D:\Hilabs -Recurse -Filter "c2b_step4_kept_prices.parquet" | Select-Object FullName

'Get-ChildItem' is not recognized as an internal or external command,
operable program or batch file.


In [28]:
import glob
print(glob.glob(r"d:\Hilabs\**\c2b_step4_kept_prices.parquet", recursive=True))

['d:\\Hilabs\\c2b_step4_kept_prices.parquet']
